In [2]:
import sys
sys.path.insert(0, '../')

# Recharge le code source après modification (évite les modules Python en cache)
for _mod in list(sys.modules):
    if _mod == 'src' or _mod.startswith('src.'):
        del sys.modules[_mod]

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from src.preprocessing import load_fraud_data, engineer_fraud_features, encode_transaction_type
from src.models import FRAUD_MODELS, train_and_evaluate, cross_validate_models
from src.utils import evaluate_classifier, show_figure
from src.display import (
    init_notebook_theme, show_hero, show_section, show_insight,
    show_info, show_warning, show_success, show_metrics_row,
    show_findings_list, show_badge, show_table_html, show_architecture_card,
)
init_notebook_theme()

DATA_PATH = '../data/raw/detection_fraude.csv'
RANDOM_STATE = 42


In [3]:
show_hero(
    title="Exercice 1 — Détection de fraude bancaire",
    objective="Développer un système de détection automatique des transactions frauduleuses.",
    plan_items=[
        "Analyse exploratoire (EDA)", "Prétraitement & feature engineering",
        "Gestion du déséquilibre", "Modélisation", "Évaluation",
        "Interprétabilité SHAP", "Analyse des erreurs",
    ],
)
show_section("1. Analyse exploratoire des données (EDA)")

In [4]:
df = load_fraud_data(DATA_PATH)
show_metrics_row({"Lignes": f"{df.shape[0]:,}", "Colonnes": df.shape[1]})
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [5]:
print("\n--- Informations générales ---")
df.info()
print("\n--- Valeurs manquantes ---")
print(df.isnull().sum())
print("\n--- Statistiques descriptives ---")
df.describe()


--- Informations générales ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 11 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   step            1048575 non-null  int64  
 1   type            1048575 non-null  object 
 2   amount          1048575 non-null  float64
 3   nameOrig        1048575 non-null  object 
 4   oldbalanceOrg   1048575 non-null  float64
 5   newbalanceOrig  1048575 non-null  float64
 6   nameDest        1048575 non-null  object 
 7   oldbalanceDest  1048575 non-null  float64
 8   newbalanceDest  1048575 non-null  float64
 9   isFraud         1048575 non-null  int64  
 10  isFlaggedFraud  1048575 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 88.0+ MB

--- Valeurs manquantes ---
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDes

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1048575.0
mean,2.696617e+01,1.586670e+05,8.740055e+05,8.938049e+05,9.781600e+05,1.114193e+06,1.089097e-03,0.0
std,1.562325e+01,2.649409e+05,2.971725e+06,3.008246e+06,2.296779e+06,2.416554e+06,3.298351e-02,0.0
min,1.000000e+00,1.000000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0
25%,1.500000e+01,1.214907e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0
50%,2.000000e+01,7.634333e+04,1.600200e+04,0.000000e+00,1.263772e+05,2.182604e+05,0.000000e+00,0.0
75%,3.900000e+01,2.137619e+05,1.366420e+05,1.746000e+05,9.159235e+05,1.149808e+06,0.000000e+00,0.0
max,9.500000e+01,1.000000e+07,3.893942e+07,3.894623e+07,4.205466e+07,4.216916e+07,1.000000e+00,0.0


In [6]:
from src.charts.fraud import build_class_distribution
from src.utils import show_figure

fig = build_class_distribution(DATA_PATH)
show_figure(fig, '../reports/figures/ex1_class_distribution.png', width=1000, height=450)


In [7]:
from src.charts.fraud import build_amount_distribution
from src.utils import show_figure

fig = build_amount_distribution(DATA_PATH)
show_figure(fig, '../reports/figures/ex1_amount_distribution.png', width=1000, height=450)


In [8]:
from src.charts.fraud import build_suspicious_behavior
from src.utils import show_figure

fig = build_suspicious_behavior(DATA_PATH)
show_figure(fig, '../reports/figures/ex1_suspicious_behavior.png', width=1000, height=450)


In [9]:
show_section("2. Prétraitement & Feature Engineering")

In [10]:
# Feature engineering
df_feat = engineer_fraud_features(df)
df_feat = encode_transaction_type(df_feat)

# Supprimer les colonnes non pertinentes (identifiants)
drop_cols = ['nameOrig', 'nameDest', 'isFlaggedFraud']
df_feat = df_feat.drop(columns=drop_cols, errors='ignore')

print("Features disponibles :")
print(df_feat.columns.tolist())
print(f"\nShape après feature engineering : {df_feat.shape}")

Features disponibles :
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'error_balance_orig', 'error_balance_dest', 'orig_zeroed', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']

Shape après feature engineering : (1048575, 14)


In [11]:
X = df_feat.drop(columns=['isFraud'])
y = df_feat['isFraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
show_metrics_row({
    "Train": f"{X_train.shape[0]:,}",
    "Fraudes train": f"{y_train.sum()} ({y_train.mean()*100:.2f}%)",
    "Test": f"{X_test.shape[0]:,}",
    "Fraudes test": f"{y_test.sum()} ({y_test.mean()*100:.2f}%)",
})

In [12]:
show_section("3. Gestion du déséquilibre des classes", "~0.1% de fraudes dans le dataset")
show_warning("Stratégies : SMOTE, class_weight='balanced', seuil de décision ajusté.")

In [13]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)
smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
before = y_train.value_counts().to_dict()
after = pd.Series(y_train_resampled).value_counts().to_dict()
show_metrics_row({
    "Normal avant SMOTE": f"{before.get(0, 0):,}",
    "Fraude avant SMOTE": f"{before.get(1, 0):,}",
    "Normal après SMOTE": f"{after.get(0, 0):,}",
    "Fraude après SMOTE": f"{after.get(1, 0):,}",
})
show_insight("SMOTE porte la classe fraude à 10% du volume normal pour stabiliser l'apprentissage.")

In [14]:
show_section("4. Modélisation")

In [15]:
show_section("Validation croisée", "ROC-AUC — 5 folds")
cv_results = cross_validate_models(FRAUD_MODELS, X_train_scaled, y_train)

logistic_regression       roc_auc: 0.9895 ± 0.0019
random_forest             roc_auc: 0.9907 ± 0.0028
xgboost                   roc_auc: 0.9961 ± 0.0033
lightgbm                  roc_auc: 0.9900 ± 0.0029


In [16]:
# Entraînement et évaluation sur le test set
from src.constants import FRAUD_THRESHOLD

results = {}
best_model_name = max(cv_results, key=lambda k: cv_results[k]["mean"])

for name, model in FRAUD_MODELS.items():
    y_pred, y_proba = train_and_evaluate(
        model, X_train_resampled, y_train_resampled, X_test_scaled, y_test, model_name=name
    )
    results[name] = evaluate_classifier(y_test, y_pred, y_proba, model_name=name)
    if name == best_model_name:
        y_proba_best = y_proba
        y_pred_best = (y_proba >= FRAUD_THRESHOLD).astype(int)

show_success(f"Meilleur modèle (CV ROC-AUC) : {best_model_name}")

,Classe,Precision,Recall,F1,Support
0,Normal,1.000,0.951,0.975,209487
1,Fraude,0.021,0.974,0.042,228


,Classe,Precision,Recall,F1,Support
0,Normal,1.000,1.000,1.000,209487
1,Fraude,0.933,0.978,0.955,228


,Classe,Precision,Recall,F1,Support
0,Normal,1.000,0.999,1.000,209487
1,Fraude,0.651,0.965,0.777,228


,Classe,Precision,Recall,F1,Support
0,Normal,1.000,1.000,1.000,209487
1,Fraude,0.820,0.978,0.892,228


In [17]:
from src.charts.fraud import build_roc_curves
from src.utils import show_figure

fig = build_roc_curves(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex1_roc_curves.png', width=900, height=600)


In [18]:
#Réseau de neurones
import tensorflow as tf
from tensorflow import keras

nn_model = keras.Sequential([
    keras.layers.Input(shape=(X_train_resampled.shape[1],)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])

nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['AUC', 'Precision', 'Recall']
)

history = nn_model.fit(
    X_train_resampled, y_train_resampled,
    epochs=20, batch_size=512,
    validation_split=0.15,
    verbose=1
)

Epoch 1/20
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - AUC: 0.7332 - Precision: 0.1748 - Recall: 0.1097 - loss: 0.0293 - val_AUC: 0.9747 - val_Precision: 1.0000 - val_Recall: 0.5697 - val_loss: 0.8731
Epoch 2/20
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - AUC: 0.9568 - Precision: 0.9127 - Recall: 0.5328 - loss: 0.0028 - val_AUC: 0.9500 - val_Precision: 1.0000 - val_Recall: 0.5511 - val_loss: 1.0018
Epoch 3/20
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - AUC: 0.9614 - Precision: 0.9095 - Recall: 0.5920 - loss: 0.0022 - val_AUC: 0.9485 - val_Precision: 1.0000 - val_Recall: 0.6375 - val_loss: 0.8815
Epoch 4/20
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - AUC: 0.9567 - Precision: 0.9149 - Recall: 0.6664 - loss: 0.0020 - val_AUC: 0.9840 - val_Precision: 1.0000 - val_Recall: 0.6546 - val_loss: 0.7667
Epoch 5/20
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - AUC: 0.9686 - Precision: 0.9104 - Recall: 0.6478 - loss: 0.0018 - val_AUC: 0.9621 - val_Precision: 1.0000 - val_Recall: 0.6524 - val_los

In [20]:
# Courbe d'apprentissage du réseau de neurones (Plotly)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.utils import show_figure

epochs = list(range(1, len(history.history['loss']) + 1))
fig = make_subplots(rows=1, cols=2, subplot_titles=['Perte (Loss)', 'AUC'])
fig.add_trace(go.Scatter(x=epochs, y=history.history['loss'], mode='lines+markers', name='Train'), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=history.history['val_loss'], mode='lines+markers', name='Validation'), row=1, col=1)
# Keras conserve la casse des métriques déclarées dans compile() → 'AUC' / 'val_AUC'
fig.add_trace(go.Scatter(x=epochs, y=history.history['AUC'], mode='lines+markers', name='Train'), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs, y=history.history['val_AUC'], mode='lines+markers', name='Validation'), row=1, col=2)
fig.update_xaxes(title_text='Epoch', row=1, col=1)
fig.update_xaxes(title_text='Epoch', row=1, col=2)
fig.update_yaxes(title_text='Loss', row=1, col=1)
fig.update_yaxes(title_text='AUC', row=1, col=2)
show_figure(fig, '../reports/figures/ex1_nn_training.png', width=1000, height=450)


In [21]:
show_section("5. Évaluation détaillée du meilleur modèle")

In [22]:
from src.charts.fraud import build_best_model_eval
from src.utils import show_figure

fig = build_best_model_eval(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex1_best_model_eval.png', width=1000, height=500)


In [23]:
show_section("6. Interprétabilité — SHAP")

In [24]:
from src.charts.fraud import build_shap_importance
from src.utils import show_figure

fig = build_shap_importance(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex1_shap_importance.png', width=900, height=600)


In [25]:
from src.charts.fraud import build_shap_beeswarm
from src.utils import show_figure

fig = build_shap_beeswarm(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex1_shap_beeswarm.png', width=900, height=700)


In [26]:
from src.charts.fraud import build_shap_force_plot
from src.utils import show_figure

fig = build_shap_force_plot(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex1_shap_force_plot.png', width=900, height=600)


In [27]:
show_section("7. Analyse des faux positifs / faux négatifs")

In [29]:
X_test_df = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)
results_df = X_test_df.copy()
results_df['y_true'] = y_test.values
results_df['y_pred'] = y_pred_best
results_df['y_proba'] = y_proba_best

faux_positifs = results_df[(results_df['y_true'] == 0) & (results_df['y_pred'] == 1)]
faux_negatifs = results_df[(results_df['y_true'] == 1) & (results_df['y_pred'] == 0)]


print("\n--- Caractéristiques des faux négatifs ---")
print(faux_negatifs.describe())

show_metrics_row({"Faux positifs": len(faux_positifs), "Faux négatifs": len(faux_negatifs)})
show_warning("Faux positifs = normales bloquées. Faux négatifs = fraudes manquées (critique).")



--- Caractéristiques des faux négatifs ---
           step    amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
count  8.000000  8.000000       8.000000        8.000000        8.000000   
mean  -0.237852  0.537544      -0.290044       -0.297188        0.655417   
std    1.288724  1.784799       0.009017        0.000000        1.229931   
min   -1.213989 -0.598889      -0.294168       -0.297188       -0.425863   
25%   -0.957953 -0.537052      -0.294168       -0.297188       -0.339366   
50%   -0.765926 -0.181259      -0.294148       -0.297188        0.272534   
75%   -0.173843  0.720004      -0.292273       -0.297188        1.266818   
max    2.434523  4.671371      -0.268654       -0.297188        3.044495   

       newbalanceDest  error_balance_orig  error_balance_dest  orig_zeroed  \
count        8.000000            8.000000            8.000000     8.000000   
mean         0.695651            0.392150            0.077019     0.643034   
std          1.119979            1.68

In [30]:
from src.charts.fraud import build_threshold_analysis
from src.utils import show_figure

fig = build_threshold_analysis(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex1_threshold_analysis.png', width=1000, height=450)


In [31]:
show_section("Synthèse Exercice 1")
show_findings_list([
    "Dataset déséquilibré (~0.1% fraudes) → SMOTE + class_weight",
    "Features error_balance_orig et orig_zeroed très discriminantes",
    f"Meilleur modèle : {best_model_name}",
    "SHAP : soldes et type de transaction dominent",
    "Seuil F1 optimal > 0.5 pour limiter les faux négatifs",
], title="Points clés")
show_insight("Privilégier le rappel fraude avec revue humaine des alertes.")